# Merlin — Pipeline Walkthrough

This notebook is an interactive walkthrough of the **Merlin** SBI pipeline for
Euclid 3x2pt cosmology, using the current `merlin` package API end to end:

1. Load a config file and inspect it (fiducial values, priors, param groups)
2. Build the `Simulator` and inspect it
3. Draw prior samples and visualize the forward model
4. Start a training run (`train_<N>/`) and generate the fiducial observation
5. Run the full pipeline (simulate → train → plot) via the CLI, or step by step
6. Load a Zarr store and preprocess it (Cholesky whitening + PCA)
7. Visualize PCA variance and parameter priors
8. Apply scale cuts
9. Train the network (manual, step-by-step alternative to `merlin-train`)
10. Run a coverage/calibration test

Replaces the old `example_pipeline.ipynb` and `showcase.ipynb`, both of which
referenced functions/config sections from a pre-refactor version of the code
that no longer exists — this merges what was distinct/useful in each into one
notebook, without duplicating what's already covered by `corner_plot.ipynb`
(loading predictions from a trained checkpoint and plotting a corner plot
against MCMC — see "Next: plotting predictions" at the end).

> **Prerequisites**: install the package from the repo root with `pip install -e .`
> (see the README), and ensure `cloelib`, `euclidlib`, and `swyft` are available
> in your environment.
>
> **Heads-up on runtime**: sections 2–3 call real `cloelib` physics per sample
> (seconds each); section 5's Fisher analysis and the simulation store can take
> much longer (the shipped example config targets `N_sims: 100000` — see the
> README, which recommends running `merlin-simulate`/`merlin-train` as batch
> jobs rather than interactively). Sections 6 onward assume `merlin-simulate`
> (which also runs Fisher first, if `PRIORS.use_Fisher_priors` is set) has
> already been run for the config below (so the simulation store already
> exists on disk) — the fiducial observation itself is regenerated fresh in
> section 4 below and again by `merlin-train`, so it doesn't need to pre-exist.

In [ ]:
### Imports
import numpy as np
import torch
import matplotlib.pyplot as plt
import swyft

import merlin
from merlin.io import load_array

## 1 · Load the config

Edit `CONFIG_PATH` below to point to your own config if needed. The cell prints
every top-level section, the fiducial values, and a summary of which parameters
are varied vs. fixed per `PRIORS` (see `merlin.resolve_priors`).

In [ ]:
CONFIG_PATH = "../input/config_example.yaml"   # ← change to your config

config = merlin.load_config(CONFIG_PATH)

print("Sections found:", list(config.keys()))
print()
print("FIDUCIAL VALUES:")
for k, v in config["FIDUCIAL VALUES"].items():
    print(f"  {k:15s} = {v}")

In [ ]:
fiducial, specs, varied_names, varied_indices = merlin.resolve_priors(config)

n_fixed = sum(1 for s in specs if s.kind == "fixed")
n_uniform = sum(1 for s in specs if s.kind == "uniform")
n_normal = sum(1 for s in specs if s.kind == "normal")
print(f"PRIORS: {len(specs)} parameters total — "
      f"{n_uniform} uniform, {n_normal} normal, {n_fixed} fixed")
print(f"Varied (selectable via TRAINING.params_to_infer): {varied_names}")
print()
print("Available PARAM_GROUPS:", list(merlin.PARAM_GROUPS.keys()))
print("Currently training/inferring:", config["TRAINING"]["params_to_infer"])

## 2 · Build the simulator and inspect it

`build_simulator` reads `PRIORS`/`AUX FILES` from the config and constructs a
`Simulator`. If `PRIORS.use_Fisher_priors` is true (as in the shipped example),
this **reads** an already-computed Fisher matrix from `PRIORS.finv_file` — it
does not compute it. `merlin-simulate` runs Fisher automatically first
whenever that flag is set; to compute it here without a full simulation run
(e.g. just to unblock this cell), call it directly:
```python
merlin.run_fisher(config)   # no-op-equivalent skip this if use_Fisher_priors is false
```

In [ ]:
sim = merlin.build_simulator(config)

print(f"n_bins        = {sim.n_bins}")
print(f"n_data        = {sim.n_data}  (WL + GGL + GCph combined)")
print(f"WL  spectra   = {len(sim.WL_keys)}")
print(f"GGL spectra   = {len(sim.GGL_keys)}")
print(f"GCph spectra  = {len(sim.GG_keys)}")
print(f"ell range     = [{sim.ells.min():.0f}, {sim.ells.max():.0f}]  ({len(sim.ells)} bins)")

## 3 · Draw prior samples and visualize the forward model

Draw a handful of parameter samples from the prior (`sim.sample_z`) and compute
the corresponding $C_\ell$ data vectors (`sim.get_sample_Cls`) to verify the
forward model. Each draw calls real `cloelib` physics, so keep `N_test` small.

In [ ]:
N_test = 5
z_samples = sim.sample_z(shape=(N_test,))   # (N_test, N_params)

print("Cosmological parameter names:", merlin.COSMO_PARAMS)
print(f"\n{N_test} prior draws (cosmo only):")
print(z_samples[:, :merlin.N_COSMO].round(4))

fig, ax = plt.subplots(figsize=(8, 4))
for z in z_samples:
    cls = sim.get_sample_Cls(z)
    n_wl = len(sim.WL_keys)
    wl_cls = cls[:n_wl * len(sim.ells)].reshape(n_wl, len(sim.ells))
    ax.plot(sim.ells, wl_cls.T, lw=0.9, alpha=0.6)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$C_\ell^{\rm WL}$")
ax.set_title(f"WL auto-spectra for {N_test} prior draws")
plt.tight_layout()
plt.show()

## 4 · Start a training run and generate the fiducial observation

Every training run gets its own numbered `train_<N>/` subfolder under
`RUN.run_dir` (checkpoint, csv logs, PCA projection, mock observation, plots)
— `populate_train_dir` creates the next free one and points
`STORES`/`OBSERVATION`/`PCA.SVD` at it. Do this once per notebook run, before
anything below that reads/writes those paths (this cell included).

`generate_observation` then runs the forward model at the fiducial cosmology
(or loads an external mock — see `MOCK_OBS` in the config) and Cholesky-whitens
it, saving into that new `train_<N>/`. It's independent of Fisher (only
*reads* `PRIORS.finv_file` if `use_Fisher_priors` is set, same as
`build_simulator` above — it never computes Fisher itself) and cheap enough to
call on every `merlin-train` run, which is exactly what `merlin-train` does at
the start of each run — so changing `MOCK_OBS`/`FIDUCIAL VALUES` and
re-running `merlin-train` tries a new observation against the same
simulations, with no need to redo Fisher or re-simulate.</cell id="e59a304a">


In [ ]:
train_id = merlin.populate_train_dir(config, CONFIG_PATH)
print(f"Created train_{train_id} under {config['RUN']['run_dir']}")

obs, obs_chol, Lfid = merlin.generate_observation(config)  # safe to re-run — see markdown above

print("Observation keys:", list(obs.keys()))
print("C_ells shape    :", obs["C_ells"].shape)

Nbin_z = config["AUX FILES"]["Nbin_z"]
N_spectra = Nbin_z * (2 * Nbin_z + 1)
N_ell = len(sim.ells)

cells_2d = obs["C_ells"].reshape(N_spectra, N_ell).T
noise_2d = obs["noise"].reshape(N_spectra, N_ell).T
whitened_2d = obs_chol["C_ells"].reshape(N_spectra, N_ell).T

offset_WL, offset_GGL, offset_GC = 0, len(sim.WL_keys), len(sim.WL_keys) + len(sim.GGL_keys)
cmap = plt.cm.viridis
colors = cmap(np.linspace(0, 1, Nbin_z))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plt.subplots_adjust(wspace=0)
for ax_idx, ax in enumerate(axes):
    for i in range(Nbin_z):
        ax.set_xscale("log")
        if ax_idx == 0:
            idx = i * Nbin_z - i * (i - 1) // 2
            spec_index = offset_WL + idx
            ax.set_ylabel(r"$C^{WL}_{ii}(\ell)$", fontsize=16)
            ax.set_yscale("log")
        elif ax_idx == 1:
            idx = i * Nbin_z + i
            spec_index = offset_GGL + idx
            ax.set_ylabel(r"$C^{XC}_{ii}(\ell)$", fontsize=16)
        else:
            idx = i * Nbin_z - i * (i - 1) // 2
            spec_index = offset_GC + idx
            ax.set_ylabel(r"$C^{GC}_{ii}(\ell)$", fontsize=16)
            ax.set_yscale("log")
        ax.plot(sim.ells, cells_2d[:, spec_index], ls="--", color=colors[i])
        ax.plot(sim.ells, cells_2d[:, spec_index] + noise_2d[:, spec_index], ls="-",
                color=colors[i], label=rf"$n_{{{i+1}}}$" if ax_idx == 1 else None)
    ax.set_xlabel(r"$\ell$", fontsize=16)
axes[1].legend(loc="upper right", ncol=2, fontsize=10)
axes[0].set_title("Raw (dashed = noiseless, solid = + noise draw)")
plt.tight_layout()
plt.show()

## 5 · Run the full pipeline

The remaining sections assume the simulation store and (for section 8) a
trained checkpoint already exist. The full pipeline, run as CLI commands
(install registers these — see the README):

| Step | Command | Notes |
|---|---|---|
| Generate simulations | `merlin-simulate config.yaml` | runs Fisher first if `PRIORS.use_Fisher_priors` is true (already done in section 2 above), then fills the store — **submit as a batch job** for large `N_sims` |
| Train + infer | `merlin-train config.yaml` | creates the next `train_<N>/`, regenerates the fiducial observation fresh into it (section 4 above), then trains + infers on it — re-run with a different config (architecture, scale cuts, `MOCK_OBS`, ...) any time to get a new `train_<N>` without re-simulating |
| Plot predictions | Open `corner_plot.ipynb` | loads a trained checkpoint from a specific `train_<N>`, no retraining needed |
| Quick diagnostic plots | `merlin-plot config.yaml --mode {corner,coverage,loss} --train-id N` | see section 10 below — `--train-id` picks which `train_<N>/` to plot, always required |

Equivalently, from Python:
```python
merlin.simulate(config)                                    # runs Fisher (if enabled) + fills the Zarr store
train_id = merlin.populate_train_dir(config, CONFIG_PATH)  # creates train_<N>/, points STORES/OBSERVATION/PCA.SVD at it
network, trainer = merlin.train(store_samples, V_proj, config)   # see section 8 below
```

## 6 · Load the simulation store and preprocess it

`preprocess` Cholesky-whitens the store, applies any `ANALYSIS VARIANTS.SCALE CUTS`
mask, optionally regenerates noise samples
(`ANALYSIS VARIANTS.regenerate_noise_samples` — see section 8 below), and
computes/loads the PCA projection — all in one call (see `merlin.preprocessing`
for the individual steps this wraps, if you need finer control).

In [ ]:
store = swyft.ZarrStore(config["SIMULATION"]["store_path"]).get_sample_store()
print(f"Store contains {len(store['z'])} simulations")
print("Store keys:", list(store.keys()))

store_samples, V_proj = merlin.preprocess(store, Lfid, config)
print(f"\nstore_samples['C_ells'] shape : {store_samples['C_ells'].shape}")
print(f"V_proj shape                  : {V_proj.shape}  →  {V_proj.shape[1]} PCA modes retained")

## 7 · Visualize PCA variance and parameter priors

Check how many modes are needed to reach 99.9% variance, and visualize the
prior distributions actually realized in the store's cosmological parameters.

In [ ]:
Cells_chol = np.asarray(store_samples["C_ells"])
n_ret = V_proj.shape[1]
fCells = torch.from_numpy(Cells_chol.reshape(len(store["z"]), -1))
q_viz = min(n_ret + 10, fCells.shape[1])

_, S, _ = torch.pca_lowrank(fCells, q=q_viz, center=True)
cumvar = (S.cumsum(0) / S.sum()).numpy() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(np.arange(1, len(cumvar) + 1), cumvar, marker="o", ms=3)
axes[0].axhline(99.9, color="tomato", ls="--", label="99.9%")
axes[0].axvline(n_ret, color="steelblue", ls="--", label=f"{n_ret} retained")
axes[0].set_xlabel("PCA modes")
axes[0].set_ylabel("Cumulative variance (%)")
axes[0].set_title("PCA compression")
axes[0].legend()

z_np = np.asarray(store["z"])[:, :merlin.N_COSMO]
z_norm = (z_np - z_np.mean(axis=0)) / z_np.std(axis=0)
for i, name in enumerate(merlin.COSMO_PARAMS):
    axes[1].hist(z_norm[:, i], bins=40, label=name, histtype="step", lw=1.5)
axes[1].set_xlabel("Normalized parameter value (σ)")
axes[1].set_ylabel("Count")
axes[1].set_title("Prior samples — cosmological parameters")
axes[1].legend(fontsize=8)

print(f"Retained {n_ret} modes → {cumvar[n_ret - 1]:.3f}% variance")
plt.tight_layout()
plt.show()

## 8 · Scale cuts and noise regeneration (`ANALYSIS VARIANTS`)

Both are applied at preprocessing time (section 6 above), not baked into the
store — so a different `train_<N>` run against the same store can use
different settings without re-simulating.

`ANALYSIS VARIANTS.SCALE CUTS` sets a per-probe $\ell_{\max}$; `make_scale_cut_mask`
builds the 0/1 mask over the flattened data vector that `preprocess`/
`preprocess_obs` apply automatically when that section is present.

`ANALYSIS VARIANTS.regenerate_noise_samples: true` discards the store's own
noise samples and draws fresh ones from the *current* `Lfid` instead (a single
batched matmul) — e.g. to retrain against a different `AUX FILES.covmat` than
the one active when the store was simulated, without re-running the C_ells
physics. Not demoed here since it just changes what `preprocess` already did
in section 6 — set it in the config and re-run that cell to see the effect.

In [ ]:
mask = merlin.make_scale_cut_mask(config)
print(f"{int(mask.sum())}/{len(mask)} data points retained under the configured scale cuts")

## 9 · Train the network (manual, step by step)

`merlin-train` does this unattended (preprocess → train → infer). Calling
`merlin.train` directly here is useful for interactive experimentation, but
for the shipped example's `N_sims`/`max_epochs` this can take a long time —
consider running `merlin-train` as a batch job instead if you just want a
trained checkpoint (`corner_plot.ipynb` can then load it via
`predict_from_checkpoint` with no retraining needed).

In [ ]:
network, trainer = merlin.train(store_samples, V_proj, config, num_workers=0)
print(f"Best checkpoint saved to: {config['STORES']['checkpoint_path']}/best.ckpt")

## 10 · Coverage test

A standard SBI calibration check: the true parameter should fall inside the
network's reported 68% credible region ~68% of the time across many held-out
simulations. `run_coverage_test` handles drawing a consistent prior, running
swyft's `test_coverage`, and saving the zz-plot. Once a checkpoint is saved to
disk, the same plot can also be produced non-interactively via
`merlin-plot config.yaml --mode coverage` (see `merlin/plotting.py`) — this
manual call is for when `trainer`/`network` are already in memory.

In [ ]:
import os

coverage_path = os.path.join(config["STORES"]["plots_dir"], "coverage.pdf")
merlin.run_coverage_test(trainer, network, store_samples, config, coverage_path)
print(f"Coverage plot saved to: {coverage_path}")

## Next: plotting predictions

Open **`corner_plot.ipynb`** (set `TRAIN_ID` there to `train_id` from section 4
above, or any previous run's) to load predictions from that checkpoint via
`predict_from_checkpoint` — no retraining needed — and plot a corner plot,
optionally against an MCMC chain.

See the [README](../README.md) for the full CLI reference and config options.